In [ ]:
import numpy as np
import pandas as pd
import os
import time
import matplotlib.pyplot as plt
import EMGFlow

file_path = 'ReceivedTofile-COM4-25_08_2026_10-32-38_PM.DAT'

with open(file_path, 'rb') as f:
    raw = f.read()

PACKET_SIZE = 98

num_packets = len(raw) // PACKET_SIZE

packets = np.frombuffer(
    raw[:num_packets * PACKET_SIZE],
    dtype=np.uint8
).reshape(num_packets, PACKET_SIZE)

print("Packets:", num_packets)
print("Shape:", packets.shape)

for i in range(10):
    print(
        f"Packet {i}: "
        f"header={packets[i, 0:2].tolist()} "
        f"bytes2-7={packets[i, 2:8].tolist()} "
        f"end={packets[i, 96:98].tolist()}"
    )

Packets: 7766
Shape: (7766, 98)
Packet 0: header=[170, 170] bytes2-7=[95, 0, 0, 17, 81, 231] end=[0, 85]
Packet 1: header=[170, 170] bytes2-7=[95, 0, 0, 17, 101, 233] end=[0, 85]
Packet 2: header=[170, 170] bytes2-7=[95, 0, 0, 17, 121, 250] end=[0, 85]
Packet 3: header=[170, 170] bytes2-7=[95, 0, 0, 17, 141, 255] end=[0, 85]
Packet 4: header=[170, 170] bytes2-7=[95, 0, 0, 17, 161, 255] end=[0, 85]
Packet 5: header=[170, 170] bytes2-7=[95, 0, 0, 17, 181, 255] end=[0, 85]
Packet 6: header=[170, 170] bytes2-7=[95, 0, 0, 17, 201, 255] end=[0, 85]
Packet 7: header=[170, 170] bytes2-7=[95, 0, 0, 17, 221, 255] end=[0, 85]
Packet 8: header=[170, 170] bytes2-7=[95, 0, 0, 17, 241, 255] end=[0, 85]
Packet 9: header=[170, 170] bytes2-7=[95, 0, 0, 18, 5, 255] end=[0, 85]


In [ ]:
data = read_emg_file(
    "ReceivedTofile-COM4-25_08_2026_10-32-38_PM.DAT"
)

print(data.shape)

(85426, 8)


In [ ]:
import pandas as pd

df = pd.DataFrame(
    data,
    columns=[
        "EMG_1",
        "EMG_2",
        "EMG_3",
        "EMG_4",
        "EMG_5",
        "EMG_6",
        "EMG_7",
        "EMG_8"
    ]
)

print(df.head())

   EMG_1  EMG_2  EMG_3  EMG_4  EMG_5  EMG_6  EMG_7  EMG_8
0    127    127    127    127    127    127    127    127
1    127    127    126    125    124    127    125    128
2    128    127    127    125    124    128    125    129
3    124    125    123    123    124    125    123    124
4    124    125    124    123    124    126    123    125


In [ ]:
import os
import time


last_size = 0
index = 0

while True:
    size = os.path.getsize(file_path)

    if size != last_size:
        print(f"{index}: File size: {size} bytes (+{size - last_size})")
        last_size = size
        index += 1

    time.sleep(0.1)

0: File size: 761165 bytes (+761165)


KeyboardInterrupt: 

In [ ]:

def dat_to_dataframe(file_path):
    
    PACKET_SIZE = 98

    with open(file_path, "rb") as f:
        raw = f.read()

    usable = (len(raw) // PACKET_SIZE) * PACKET_SIZE
    raw = raw[:usable]

    packets = np.frombuffer(
        raw,
        dtype=np.uint8
    ).reshape(-1, PACKET_SIZE)

    # 88-byte payload
    payload = packets[:, 8:96]

    # 11 samples × 8 channels
    data = payload.reshape(-1, 8)

    # Centre around 127
    data = data.astype(float) - 127

    df = pd.DataFrame(
        data,
        columns=[
            "EMG_1",
            "EMG_2",
            "EMG_3",
            "EMG_4",
            "EMG_5",
            "EMG_6",
            "EMG_7",
            "EMG_8"
        ]
    )

    return df

In [ ]:
df = dat_to_dataframe(
    "ReceivedTofile-COM4-25_08_2026_10-32-38_PM.DAT"
)

print(df.shape)
print(df.head())

(85426, 8)
   EMG_1  EMG_2  EMG_3  EMG_4  EMG_5  EMG_6  EMG_7  EMG_8
0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0
1    0.0    0.0   -1.0   -2.0   -3.0    0.0   -2.0    1.0
2    1.0    0.0    0.0   -2.0   -3.0    1.0   -2.0    2.0
3   -3.0   -2.0   -4.0   -4.0   -3.0   -2.0   -4.0   -3.0
4   -3.0   -2.0   -3.0   -4.0   -3.0   -1.0   -4.0   -2.0


In [ ]:
EMGFlow.apply_notch_filters('EMG_1')
EMGFlow.apply_bandpass_filter('EMG_1')
EMGFlow.apply_rms_smooth('EMG_1')

TypeError: apply_notch_filters() missing 1 required positional argument: 'column_name'